# 🎤 RVC-Python: Voice Conversion Demo

This notebook demonstrates **RVC (Retrieval-based Voice Conversion)** using the `rvc-python` library.

### What you can do:
- Download RVC base models and custom voice models from **any link**
- Upload your own audio and a pre-trained RVC model
- Convert voices with different pitch extraction methods
- Adjust pitch, protection, and other parameters
- Run everything on a free Google Colab GPU

### Supported download sources:
- 🔗 **Direct URL** (`.pth`, `.index`, `.pt`)
- 🤗 **HuggingFace** (auto-converts to raw download URL)
- 📁 **Google Drive** (file sharing links & folders)
- 📦 **Any public link** (Mega, Dropbox, etc.)

### Requirements:
- A **.pth** RVC model file (v1 or v2)
- An audio file to convert (.wav, .mp3, .flac, etc.)
- Optional: a **.index** file for better voice similarity

---

## 1️⃣ Check GPU & Install Dependencies

Make sure you're using a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [ ]:
!nvidia-smi -L

In [ ]:
# Install PyTorch with CUDA 12.1 support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

# Install rvc-python from source (with Python 3.12+ support)
!pip install git+https://github.com/onxlmao/rvc-python.git -q

# Install download helpers
!pip install gdown -q

print("✅ Installation complete!")

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2️⃣ Download Models

You can download RVC models from **any link**. The notebook auto-detects the source type.

### Supported link formats:

| Source | Example |
|--------|----------|
| **Direct URL** | `https://example.com/model.pth` |
| **HuggingFace** | `https://huggingface.co/user/repo/blob/main/model.pth` |
| **HuggingFace (raw)** | `https://huggingface.co/user/repo/resolve/main/model.pth` |
| **Google Drive** | `https://drive.google.com/file/d/FILE_ID/view` |
| **Google Drive (folder)** | `https://drive.google.com/drive/folders/FOLDER_ID` |

In [ ]:
import os

# Create directories
os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("base_model", exist_ok=True)

print("📁 Directories created:")
print("  - input/       (put your audio files here)")
print("  - output/      (converted files will appear here)")
print("  - models/      (put your .pth and .index files here)")
print("  - base_model/  (RVC base models: hubert, rmvpe)")

### 📦 Download Base Models (Required)

These base models are **required** for RVC inference (HuBERT + RMVPE).

In [ ]:
# @title Download RVC Base Models (HuBERT + RMVPE)
import requests
import os

BASE_MODEL_DIR = "base_model"

base_files = {
    "hubert_base.pt": "https://huggingface.co/Daswer123/RVC_Base/resolve/main/hubert_base.pt",
    "rmvpe.pt": "https://huggingface.co/Daswer123/RVC_Base/resolve/main/rmvpe.pt",
    "rmvpe.onnx": "https://huggingface.co/Daswer123/RVC_Base/resolve/main/rmvpe.onnx",
}

print("📦 Downloading RVC base models...")
print("=" * 50)

for filename, url in base_files.items():
    filepath = os.path.join(BASE_MODEL_DIR, filename)
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"✅ {filename} already exists ({size_mb:.1f} MB)")
        continue
    
    print(f"⬇️  Downloading {filename}...")
    try:
        r = requests.get(url, stream=True, timeout=60)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(filepath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total > 0:
                    pct = downloaded / total * 100
                    print(f"\r   Progress: {pct:.0f}% ({downloaded/1024/1024:.1f}/{total/1024/1024:.1f} MB)", end="")
        print(f"\n   ✅ Saved to {filepath}")
    except Exception as e:
        print(f"\n   ❌ Failed to download {filename}: {e}")

print("\n" + "=" * 50)
print("✅ Base model download complete!")

### 🎤 Download Custom Voice Model from Any Link

Paste **any link** below — the notebook will auto-detect the source and download it.

Supported formats:
- **Direct URL**: ends with `.pth`, `.index`, `.zip`, `.tar.gz`
- **HuggingFace**: any HF page URL (auto-converted to raw download)
- **Google Drive**: sharing links with `/file/d/` or `/drive/folders/`
- **ZIP archives**: will be auto-extracted to the `models/` folder

In [ ]:
# @title 📥 Download Model from Any Link
import re
import os
import zipfile
import tarfile
import requests
import shutil
from urllib.parse import urlparse

# @markdown Paste your model link below:
MODEL_LINK = ""  # @param {type:"string"}
# @markdown Model name (leave empty to auto-detect from filename):
MODEL_NAME = ""  # @param {type:"string"}

def detect_source(url):
    """Detect the download source type from URL."""
    url_lower = url.lower()
    
    # Google Drive file
    if "drive.google.com/file/d/" in url_lower or "drive.google.com/file/d/" in url:
        return "gdrive_file"
    # Google Drive folder
    if "drive.google.com/drive/folders/" in url_lower:
        return "gdrive_folder"
    # HuggingFace (any page, blob, resolve, tree)
    if "huggingface.co" in url_lower:
        return "huggingface"
    # Direct download
    return "direct"

def extract_gdrive_id(url):
    """Extract file or folder ID from Google Drive URL."""
    # File: https://drive.google.com/file/d/FILE_ID/view?usp=sharing
    m = re.search(r'/file/d/([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1), "file"
    # Folder: https://drive.google.com/drive/folders/FOLDER_ID
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1), "folder"
    # Open/d/ format
    m = re.search(r'open\?id=([a-zA-Z0-9_-]+)', url)
    if m:
        return m.group(1), "file"
    return None, None

def convert_hf_to_download_url(url):
    """Convert any HuggingFace URL to a raw download URL."""
    # https://huggingface.co/user/repo/blob/main/path/to/file.pth
    # -> https://huggingface.co/user/repo/resolve/main/path/to/file.pth
    url_converted = re.sub(r'/blob/', '/resolve/', url)
    # https://huggingface.co/user/repo/tree/main/path/
    # -> We'll list the tree page to find files
    if '/tree/' in url_converted:
        return url_converted, "tree"
    return url_converted, "file"

def guess_filename_from_url(url, source_type):
    """Try to guess the output filename from URL."""
    if source_type == "huggingface":
        # /resolve/main/model.pth -> model.pth
        m = re.search(r'/resolve/main/(.+)', url)
        if m:
            return m.group(1).split('/')[-1]
    
    # Try from URL path
    parsed = urlparse(url)
    basename = os.path.basename(parsed.path)
    if basename and '.' in basename:
        return basename
    
    return None

def extract_archive(filepath, dest_dir):
    """Extract ZIP or TAR.GZ archive to dest_dir."""
    print(f"📦 Extracting archive...")
    if filepath.endswith('.zip'):
        with zipfile.ZipFile(filepath, 'r') as zf:
            zf.extractall(dest_dir)
    elif filepath.endswith(('.tar.gz', '.tgz')):
        with tarfile.open(filepath, 'r:gz') as tf:
            tf.extractall(dest_dir)
    elif filepath.endswith('.tar'):
        with tarfile.open(filepath, 'r') as tf:
            tf.extractall(dest_dir)
    else:
        return False
    return True

def download_with_progress(url, filepath):
    """Download a file with progress bar."""
    r = requests.get(url, stream=True, timeout=120, allow_redirects=True)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(filepath, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total > 0:
                pct = downloaded / total * 100
                bar_len = 30
                filled = int(bar_len * pct / 100)
                bar = '█' * filled + '░' * (bar_len - filled)
                print(f"\r   [{bar}] {pct:.0f}% ({downloaded/1024/1024:.1f}/{total/1024/1024:.1f} MB)", end="")
    print()
    return filepath

def find_model_files(directory):
    """Find .pth and .index files in a directory (recursive)."""
    pth_files = []
    index_files = []
    for root, dirs, files in os.walk(directory):
        for f in files:
            full = os.path.join(root, f)
            if f.endswith('.pth'):
                pth_files.append(full)
            elif f.endswith('.index'):
                index_files.append(full)
    return pth_files, index_files

# ====== MAIN DOWNLOAD LOGIC ======

if not MODEL_LINK.strip():
    print("⚠️  No link provided. Skipping download.")
    print("   You can upload models manually in the next section.")
    MODEL_PATH = ""
    INDEX_PATH = ""
    
else:
    source = detect_source(MODEL_LINK.strip())
    print(f"🔍 Detected source type: {source}")
    print(f"🔗 Link: {MODEL_LINK.strip()}")
    print()
    
    MODEL_PATH = ""
    INDEX_PATH = ""
    
    # ----- Google Drive -----
    if source == "gdrive_file":
        file_id, _ = extract_gdrive_id(MODEL_LINK.strip())
        if file_id:
            gdrive_url = f"https://drive.google.com/uc?export=download&id={file_id}&confirm=t"
            filename = MODEL_NAME if MODEL_NAME else "gdrive_model.pth"
            filepath = os.path.join("models", filename)
            print(f"⬇️  Downloading from Google Drive...")
            try:
                download_with_progress(gdrive_url, filepath)
                print(f"✅ Saved to {filepath}")
                
                # Check if it's an archive
                if filepath.endswith('.zip') or filepath.endswith('.tar.gz'):
                    extract_archive(filepath, "models")
                    os.remove(filepath)
                    pth_files, idx_files = find_model_files("models")
                    if pth_files:
                        MODEL_PATH = pth_files[0]
                        print(f"🎯 Model found: {MODEL_PATH}")
                    if idx_files:
                        INDEX_PATH = idx_files[0]
                        print(f"🎯 Index found: {INDEX_PATH}")
                else:
                    MODEL_PATH = filepath
            except Exception as e:
                print(f"❌ Google Drive download failed: {e}")
                print("   Trying with gdown...")
                try:
                    import gdown
                    output = gdown.download(f"https://drive.google.com/uc?id={file_id}", "models/", quiet=False, fuzzy=True)
                    if output:
                        print(f"✅ Downloaded via gdown: {output}")
                        MODEL_PATH = output
                except Exception as e2:
                    print(f"❌ gdown also failed: {e2}")
    
    elif source == "gdrive_folder":
        folder_id, _ = extract_gdrive_id(MODEL_LINK.strip())
        if folder_id:
            print(f"⬇️  Downloading Google Drive folder...")
            try:
                import gdown
                output_dir = gdown.download_folder(f"https://drive.google.com/drive/folders/{folder_id}", output="models/", quiet=False)
                if output_dir:
                    print(f"✅ Folder downloaded to models/")
                    pth_files, idx_files = find_model_files("models")
                    if pth_files:
                        MODEL_PATH = pth_files[0]
                        print(f"🎯 Model found: {MODEL_PATH}")
                    else:
                        print("⚠️  No .pth file found in the downloaded folder.")
                    if idx_files:
                        INDEX_PATH = idx_files[0]
                        print(f"🎯 Index found: {INDEX_PATH}")
                else:
                    print("❌ Failed to download folder.")
            except ImportError:
                print("❌ gdown not installed. Run: !pip install gdown")
            except Exception as e:
                print(f"❌ Google Drive folder download failed: {e}")
    
    # ----- HuggingFace -----
    elif source == "huggingface":
        download_url, hf_type = convert_hf_to_download_url(MODEL_LINK.strip())
        
        if hf_type == "tree":
            # Tree page — try to scrape for downloadable files
            print("📂 HuggingFace folder detected. Looking for model files...")
            try:
                # Try listing files via HF API
                repo_match = re.match(r'https://huggingface\.co/([^/]+)/([^/]+)', MODEL_LINK.strip())
                if repo_match:
                    org, repo = repo_match.group(1), repo_match.group(2)
                    api_url = f"https://huggingface.co/api/models/{org}/{repo}"
                    resp = requests.get(api_url, timeout=30)
                    if resp.status_code == 200:
                        siblings = resp.json().get('siblings', [])
                        downloadable = [s['rfilename'] for s in siblings if not s['rfilename'].endswith('/')]
                        print(f"   Found {len(downloadable)} files in repo:")
                        for f in downloadable:
                            print(f"     - {f}")
                        
                        # Download .pth and .index files
                        for fname in downloadable:
                            if fname.endswith(('.pth', '.index', '.pt', '.onnx')):
                                furl = f"https://huggingface.co/{org}/{repo}/resolve/main/{fname}"
                                fpath = os.path.join("models", fname)
                                if not os.path.exists(fpath):
                                    print(f"\n⬇️  Downloading {fname}...")
                                    try:
                                        download_with_progress(furl, fpath)
                                        print(f"   ✅ Saved to {fpath}")
                                    except Exception as e:
                                        print(f"   ❌ Failed: {e}")
                                else:
                                    print(f"   ✅ {fname} already exists")
                        
                        pth_files, idx_files = find_model_files("models")
                        if pth_files:
                            MODEL_PATH = pth_files[0]
                            print(f"\n🎯 Model: {MODEL_PATH}")
                        if idx_files:
                            INDEX_PATH = idx_files[0]
                            print(f"🎯 Index: {INDEX_PATH}")
                    else:
                        print(f"❌ Could not list repo files (status {resp.status_code})")
                        print("   Try using a direct link to the specific file.")
                else:
                    print("❌ Could not parse HuggingFace repo URL.")
                    print("   Use format: https://huggingface.co/user/repo")
            except Exception as e:
                print(f"❌ HuggingFace folder listing failed: {e}")
        
        else:
            # Direct file URL
            guessed_name = guess_filename_from_url(download_url, "huggingface")
            filename = MODEL_NAME if MODEL_NAME else (guessed_name or "hf_model.pth")
            filepath = os.path.join("models", filename)
            
            print(f"⬇️  Downloading from HuggingFace...")
            print(f"   URL: {download_url}")
            try:
                download_with_progress(download_url, filepath)
                print(f"✅ Saved to {filepath}")
                
                # Check if archive
                if filepath.endswith('.zip') or filepath.endswith('.tar.gz'):
                    extract_archive(filepath, "models")
                    os.remove(filepath)
                    pth_files, idx_files = find_model_files("models")
                    if pth_files:
                        MODEL_PATH = pth_files[0]
                        print(f"🎯 Model found: {MODEL_PATH}")
                    if idx_files:
                        INDEX_PATH = idx_files[0]
                        print(f"🎯 Index found: {INDEX_PATH}")
                else:
                    MODEL_PATH = filepath
            except Exception as e:
                print(f"❌ HuggingFace download failed: {e}")
    
    # ----- Direct URL -----
    else:
        guessed_name = guess_filename_from_url(MODEL_LINK.strip(), "direct")
        filename = MODEL_NAME if MODEL_NAME else (guessed_name or "downloaded_model.pth")
        filepath = os.path.join("models", filename)
        
        print(f"⬇️  Downloading from direct URL...")
        try:
            download_with_progress(MODEL_LINK.strip(), filepath)
            print(f"✅ Saved to {filepath}")
            
            # Check if archive
            if filepath.endswith('.zip') or filepath.endswith('.tar.gz'):
                extract_archive(filepath, "models")
                os.remove(filepath)
                pth_files, idx_files = find_model_files("models")
                if pth_files:
                    MODEL_PATH = pth_files[0]
                    print(f"🎯 Model found: {MODEL_PATH}")
                if idx_files:
                    INDEX_PATH = idx_files[0]
                    print(f"🎯 Index found: {INDEX_PATH}")
            else:
                MODEL_PATH = filepath
        except Exception as e:
            print(f"❌ Direct download failed: {e}")

# Summary
print("\n" + "=" * 50)
print("📋 Summary:")
print(f"   Model (.pth):  {MODEL_PATH if MODEL_PATH else '(not set)'}")
print(f"   Index (.index): {INDEX_PATH if INDEX_PATH else '(not set)'}")
print("=" * 50)

In [ ]:
# @title 📥 Download Index File from Any Link (Optional)
# @markdown Paste your .index file link below (leave empty to skip):
INDEX_LINK = ""  # @param {type:"string"}

if INDEX_LINK.strip() and not INDEX_PATH:
    source = detect_source(INDEX_LINK.strip())
    print(f"🔍 Detected source type: {source}")
    
    if source == "huggingface":
        download_url, _ = convert_hf_to_download_url(INDEX_LINK.strip())
    else:
        download_url = INDEX_LINK.strip()
    
    guessed_name = guess_filename_from_url(download_url, source)
    filename = guessed_name if guessed_name else "added_index.index"
    filepath = os.path.join("models", filename)
    
    print(f"⬇️  Downloading index file...")
    try:
        download_with_progress(download_url, filepath)
        INDEX_PATH = filepath
        print(f"✅ Index saved to {INDEX_PATH}")
    except Exception as e:
        print(f"❌ Download failed: {e}")
elif INDEX_PATH:
    print(f"✅ Index already set: {INDEX_PATH}")
else:
    print("⏭️  No index link provided. Index will be empty.")

print(f"\n📌 Index path: {INDEX_PATH if INDEX_PATH else '(none)'}")

### 📁 List Downloaded Models

Verify what's in your `models/` and `base_model/` folders.

In [ ]:
# @title List all model files
import os

print("📂 models/")
print("-" * 40)
if os.path.exists("models"):
    items = os.listdir("models")
    if items:
        for f in sorted(items):
            size = os.path.getsize(os.path.join("models", f))
            print(f"  {f}  ({size/1024/1024:.1f} MB)")
    else:
        print("  (empty)")

print(f"\n📂 base_model/")
print("-" * 40)
if os.path.exists("base_model"):
    items = os.listdir("base_model")
    if items:
        for f in sorted(items):
            size = os.path.getsize(os.path.join("base_model", f))
            print(f"  {f}  ({size/1024/1024:.1f} MB)")
    else:
        print("  (empty)")

## 3️⃣ Upload Your Files (Alternative)

If you prefer to upload manually instead of downloading from a link, use the cells below:

1. **Model file** (`.pth`) — Your trained RVC model
2. **Index file** (`.index`, optional) — FAISS index for better similarity
3. **Input audio** (`.wav`, `.mp3`, etc.) — The audio you want to convert

In [ ]:
# @title Upload Model File (.pth) - Skip if already downloaded
from google.colab import files
import shutil

if MODEL_PATH and os.path.exists(MODEL_PATH):
    print(f"✅ Model already set: {MODEL_PATH}")
    print("   (Skipping upload. Clear MODEL_PATH to re-upload.)")
else:
    print("📁 Select your .pth model file:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        shutil.move(filename, f"models/{filename}")
        print(f"✅ Saved to models/{filename}")

    MODEL_PATH = f"models/{list(uploaded.keys())[0]}"
    print(f"\n📌 Model path: {MODEL_PATH}")

In [ ]:
# @title Upload Index File (.index) - Skip if already downloaded
from google.colab import files
import shutil

if INDEX_PATH and os.path.exists(INDEX_PATH):
    print(f"✅ Index already set: {INDEX_PATH}")
    print("   (Skipping upload. Clear INDEX_PATH to re-upload.)")
else:
    INDEX_PATH = ""
    try:
        print("📁 Select your .index file (or cancel to skip):")
        uploaded = files.upload()
        
        for filename in uploaded.keys():
            shutil.move(filename, f"models/{filename}")
            print(f"✅ Saved to models/{filename}")
        
        INDEX_PATH = f"models/{list(uploaded.keys())[0]}"
        print(f"\n📌 Index path: {INDEX_PATH}")
    except:
        print("⏭️ Skipped — no index file loaded.")

In [ ]:
# @title Upload Audio to Convert
from google.colab import files
import shutil

print("📁 Select your audio file (.wav, .mp3, etc.):")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"input/{filename}")
    print(f"✅ Saved to input/{filename}")

INPUT_PATH = f"input/{list(uploaded.keys())[0]}"
print(f"\n📌 Input path: {INPUT_PATH}")

## 4️⃣ Configure & Run Voice Conversion

Adjust the parameters below and run the conversion.

In [ ]:
# @title Conversion Settings
PITCH_SHIFT = 0          # @param {type:"slider", min:-12, max:12, step:1}
PITCH_METHOD = "rmvpe"  # @param ["rmvpe", "harvest", "crepe", "pm"]
INDEX_RATE = 0.5         # @param {type:"slider", min:0, max:1, step:0.05}
FILTER_RADIUS = 3        # @param {type:"slider", min:0, max:7, step:1}
PROTECT = 0.33           # @param {type:"slider", min:0, max:0.5, step:0.01}
RMS_MIX_RATE = 1.0       # @param {type:"slider", min:0, max:1, step:0.05}
RESAMPLE_SR = 0          # @param {type:"slider", min:0, max:48000, step:1000}
MODEL_VERSION = "v2"     # @param ["v1", "v2"]

print("⚙️ Settings:")
print(f"  Pitch shift:    {PITCH_SHIFT} semitones")
print(f"  Pitch method:   {PITCH_METHOD}")
print(f"  Index rate:     {INDEX_RATE}")
print(f"  Filter radius:  {FILTER_RADIUS}")
print(f"  Protect:        {PROTECT}")
print(f"  RMS mix rate:   {RMS_MIX_RATE}")
print(f"  Resample SR:    {RESAMPLE_SR if RESAMPLE_SR > 0 else 'original'}")
print(f"  Model version:  {MODEL_VERSION}")

In [ ]:
# @title Run Voice Conversion
import time
from rvc_python.infer import RVCInference

if not MODEL_PATH or not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH}\n"
        "Please download or upload a .pth model file first."
    )

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"
print(f"🔧 Initializing RVC on {device}...")

t0 = time.time()
rvc = RVCInference(device=device)
print(f"✅ RVC initialized in {time.time() - t0:.1f}s")

print(f"\n📦 Loading model: {MODEL_PATH}")
t1 = time.time()
index = INDEX_PATH if INDEX_PATH and os.path.exists(INDEX_PATH) else ""
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=index)
print(f"✅ Model loaded in {time.time() - t1:.1f}s")

print(f"\n⚙️ Setting parameters...")
rvc.set_params(
    f0up_key=PITCH_SHIFT,
    f0method=PITCH_METHOD,
    index_rate=INDEX_RATE,
    filter_radius=FILTER_RADIUS,
    protect=PROTECT,
    rms_mix_rate=RMS_MIX_RATE,
    resample_sr=RESAMPLE_SR,
)

input_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]
output_path = f"output/{input_name}_converted.wav"

print(f"\n🎤 Converting: {INPUT_PATH}")
print(f"   Output:   {output_path}")
t2 = time.time()
rvc.infer_file(INPUT_PATH, output_path)
elapsed = time.time() - t2
print(f"✅ Conversion complete in {elapsed:.1f}s")

# Get file size
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"📁 Output file: {output_path} ({size_mb:.1f} MB)")

## 5️⃣ Listen & Download

In [ ]:
# @title Play the converted audio
from IPython.display import Audio, display

print("🔊 Converted audio:")
display(Audio(output_path))

In [ ]:
# @title Play the original audio (for comparison)
from IPython.display import Audio, display

print("🔊 Original audio:")
display(Audio(INPUT_PATH))

In [ ]:
# @title Download converted audio
from google.colab import files

print("💾 Downloading converted audio...")
files.download(output_path)

## 6️⃣ Batch Conversion (Optional)

Upload multiple audio files to `input/` and convert them all at once.

In [ ]:
# @title Upload multiple audio files
from google.colab import files
import shutil

print("📁 Select multiple audio files to convert:")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"input/{filename}")
    print(f"✅ Saved to input/{filename}")

print(f"\n📂 {len(uploaded)} file(s) uploaded")

In [ ]:
# @title Run batch conversion
import time
from rvc_python.infer import RVCInference

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"

rvc = RVCInference(device=device)
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=index)
rvc.set_params(
    f0up_key=PITCH_SHIFT,
    f0method=PITCH_METHOD,
    index_rate=INDEX_RATE,
    filter_radius=FILTER_RADIUS,
    protect=PROTECT,
    rms_mix_rate=RMS_MIX_RATE,
    resample_sr=RESAMPLE_SR,
)

t0 = time.time()
results = rvc.infer_dir("input", "output")
elapsed = time.time() - t0

print(f"✅ Converted {len(results)} files in {elapsed:.1f}s")
for r in results:
    size_mb = os.path.getsize(r) / 1024 / 1024
    print(f"  📁 {r} ({size_mb:.1f} MB)")

In [ ]:
# @title Download all converted files as ZIP
import zipfile
from google.colab import files

zip_path = "converted_audio.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk("output"):
        for file in files_list:
            if file.endswith('.wav'):
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, "output")
                zf.write(file_path, arcname)

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"📦 Created {zip_path} ({size_mb:.1f} MB)")
files.download(zip_path)

## 7️⃣ Experiment with Different Settings

Try different pitch shifts and methods to find the best result.

In [ ]:
# @title Quick A/B comparison with different pitch shifts
from IPython.display import Audio, display
from rvc_python.infer import RVCInference
import time

device = "cuda:0" if torch.cuda.is_available() else "cpu:0"

rvc = RVCInference(device=device)
rvc.load_model(MODEL_PATH, version=MODEL_VERSION, index_path=index)

# Convert with different pitch shifts
pitch_shifts = [-3, 0, 3, 5]
input_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]

for shift in pitch_shifts:
    rvc.set_params(f0up_key=shift, f0method=PITCH_METHOD)
    out = f"output/{input_name}_pitch{shift:+d}.wav"
    
    t0 = time.time()
    rvc.infer_file(INPUT_PATH, out)
    elapsed = time.time() - t0
    
    print(f"🎵 Pitch {shift:+d} semitones ({elapsed:.1f}s):")
    display(Audio(out))
    print()

---

## Tips

| Parameter | Recommendation |
|-----------|---------------|
| **Pitch method** | Use `rmvpe` for best quality. `crepe` can be better for clean speech but is slower. |
| **Index rate** | Set to `0.5–0.8` if you have an index file. Set to `0` to ignore the index. |
| **Protect** | Keep at `0.33–0.5` to preserve breath sounds and consonants. |
| **Pitch shift** | `+12` = one octave up, `-12` = one octave down. |
| **GPU memory** | If you get OOM errors, restart the runtime and try CPU mode (`cpu:0`). |

## Troubleshooting

- **CUDA out of memory**: Use a smaller model or switch to CPU
- **Bad audio quality**: Try a different pitch method or adjust the index rate
- **Wrong voice**: Make sure you uploaded the correct `.pth` model file
- **Install errors**: Make sure you're using a GPU runtime and re-run the install cell
- **Download failed**: Check your link, try a different source, or upload manually
- **Google Drive quota**: Large files may hit download limits — try using gdown or direct link

## Links

- **GitHub**: [onxlmao/rvc-python](https://github.com/onxlmao/rvc-python)
- **Issues**: [Report a bug](https://github.com/onxlmao/rvc-python/issues)